# 06 · Funciones de orden superior: `map`, `filter`, `flatMap` y compañía

> **En el examen:** aparece como último apartado del Ejercicio 3 o 4: *"implementa la
> función **sin recursión**, usando `map`, `filter`, `flatMap`…"* o *"combinando
> exclusivamente las funciones X, Y y Z"*. También hay ejercicios enteros de "cadenas de
> procesamiento" (números romanos, binario, combinaciones, consultas de películas).
> Suelen valer 0,5-2 puntos y son de los más rápidos si te sabes el catálogo.

Requisito: archivos 01 y 04.

---

### Preámbulo
Ejecuta la siguiente celda antes que las demás. Las celdas de código se pueden ejecutar en orden; los fragmentos que no son ejecutables por sí solos se muestran como texto.

In [ ]:
// Preámbulo: ejecuta esta celda primero (las mismas definiciones que vienen en los exámenes)
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A
    def equalA(a1: A, a2: A): Boolean = a1 == a2
    def equalB(b1: B, b2: B): Boolean = b1 == b2
    def law1(a: A): Boolean = equalA(to(from(a)), a)
    def law2(b: B): Boolean = equalB(from(to(b)), b)

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)
    def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
        tree match
            case Empty() => empty
            case Node(left, a, right) => node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

import Tree._

def pow(base: Int, exp: Int): Int = math.pow(base, exp).toInt

## 1. ¿Qué es una función de orden superior (HOF)?

Una función que **recibe otra función como argumento** (o la devuelve). `foldRight` y
`foldLeft` ya lo eran. Las de este archivo resuelven patrones muy comunes **sin escribir
recursión**: tú solo dices *qué* hacer con cada elemento.

---

## 2. El catálogo (lo que tienes que saberte)

Las signaturas que vienen en el preámbulo del examen:

In [ ]:
def map[B](f: A => B): List[B]
def filter(f: A => Boolean): List[A]
def flatMap[B](f: A => List[B]): List[B]
def forall(pred: A => Boolean): Boolean
def exists(pred: A => Boolean): Boolean

### `map`: transformar cada elemento

```scala
List(1, 2, 3).map(x => x * 10)        // List(10, 20, 30)
List(1, 2, 3).map(_ * 10)             // lo mismo
List("a", "bb").map(_.length)         // List(1, 2)
```

Misma longitud que la original; cada elemento pasa por `f`.

### `filter`: quedarse con los que cumplen

```scala
List(1, 2, 3, 4).filter(_ % 2 == 0)   // List(2, 4)
```

### `flatMap`: cada elemento produce **una lista**, y se juntan todas

```scala
List(1, 2, 3).flatMap(x => List(x, x))      // List(1, 1, 2, 2, 3, 3)
List(1, 2, 3).flatMap(x => if x > 1 then List(x) else Nil)   // List(2, 3)
```

`flatMap` es como `map` seguido de `flatten`. Con él se puede hacer `map` (devolver
`List(f(x))`) y `filter` (devolver `List(x)` o `Nil`) a la vez.

Truco: si `f` devuelve un `Option`, `l.flatMap(f)` también funciona en la librería de Scala
(los `Some` se quedan, los `None` desaparecen).

### `exists` y `forall`

```scala
List(1, 2, 3).exists(_ > 2)     // true  (¿alguno cumple?)
List(1, 2, 3).forall(_ > 0)     // true  (¿todos cumplen?)
List().forall(_ > 100)          // true  (vacía: "todos" cumplen)
```

### Otras que salen

| Función | Qué hace | Ejemplo |
|---|---|---|
| `flatten` | junta una lista de listas | `List(List(1,2), List(3)).flatten == List(1,2,3)` |
| `sum` | suma los elementos | `List(1,2,3).sum == 6` |
| `reduce(f)` | combina todos con `f` (lista no vacía) | `List(1,2,3).reduce(_ + _) == 6` |
| `sortWith(menor)` | ordena | `List(3,1,2).sortWith(_ < _) == List(1,2,3)` |
| `take(n)` | los n primeros | `List(1,2,3).take(2) == List(1,2)` |
| `headOption` | primer elemento si existe | `List(5,6).headOption == Some(5)` |
| `reverse` | da la vuelta | |
| `zipWithIndex` | empareja con la posición | `List('a','b').zipWithIndex == List(('a',0), ('b',1))` |
| `zip(otra)` | empareja posición a posición | `List(1,2).zip(List('a','b')) == List((1,'a'), (2,'b'))` |
| `sliding(n)` | ventanas de n consecutivos | `List(1,2,3).sliding(2).toList == List(List(1,2), List(2,3))` |
| `appended(x)` / `:+ x` | añade al final | `List(1).appended(2) == List(1,2)` |
| `contains(x)` | ¿está x? | |
| `splitAt(n)` | parte en la posición n | `List(1,2,3).splitAt(1) == (List(1), List(2,3))` |
| `grouped(n)` | trozos de n | `List(1,2,3).grouped(2).toList == List(List(1,2), List(3))` |

`sliding` y `grouped` devuelven un **iterador**: añade `.toList` si quieres una lista.
Un `String` se convierte en lista de caracteres con `.toList`; un rango con `(1 to n).toList`.

### `map` en `Option` y `Either`

```scala
Some(3).map(_ + 1)             // Some(4)
(None: Option[Int]).map(_ + 1) // None
Right(3).map(_ + 1)            // Right(4)
Left("err").map(...)           // Left("err") sin tocar
```

---

## 3. Encadenar: pipelines

Lo potente es **encadenar** varias. Se lee de arriba abajo como una receta:

```scala
List(1, 2, 3, 4, 5, 6)
    .filter(_ % 2 == 0)     // List(2, 4, 6)
    .map(_ * 10)            // List(20, 40, 60)
    .sum                    // 120
```

Se puede poner cada paso en una línea empezando por `.`.

**Método para resolver un ejercicio de este tipo:** escribe en papel cómo cambia el dato
en cada paso con el ejemplo del enunciado, y elige la función del catálogo que hace cada
transformación.

---

## 4. Los `_` y las tuplas (cuidado)

Cuando la lista es de **tuplas**, en Scala 3 una lambda de dos parámetros se aplica
desempaquetando la tupla:

```scala
List((1, 0), (1, 1)).map((b, i) => b * 10 + i)   // List(10, 11)
List((1, 0), (1, 1)).map(_ * 10 + _)             // lo mismo: 1er _ = b, 2º _ = i
```

Si te lías, la forma más clara (y que funciona también en Scala 2) es con `case`:

```scala
List((1, 0), (1, 1)).map { case (b, i) => b * 10 + i }
```

Y si solo quieres un componente: `.map(_._1)`.

---

## 5. Ejemplos resueltos de exámenes

### `fromBits` con `reverse`, `zipWithIndex`, `map`, `pow` y `sum` (25-26 Extraordinaria)

Fórmula: invertir la lista y sumar `bit · 2^posición`.

```
List(1, 0, 1, 1)
  .reverse          → List(1, 1, 0, 1)
  .zipWithIndex     → List((1,0), (1,1), (0,2), (1,3))
  .map(b·2^i)       → List(1, 2, 0, 8)
  .sum              → 11
```

In [ ]:
def fromBits(l: List[Int]): Int =
    l.reverse.zipWithIndex.map((b, i) => b * pow(2, i)).sum

// versión del profesor, con placeholders:
//  l.reverse.zipWithIndex.map(_ * pow(2, _)).sum

### Números romanos con `map`, `appended`, `sliding`, `map`, `reduce` (24-25 Extraordinaria)

Idea: traducir cada símbolo a número; un número se **resta** si el siguiente es mayor.

```
"MCMXXIV".toList   → List('M','C','M','X','X','I','V')
  .map(Sym)        → List(1000, 100, 1000, 10, 10, 1, 5)
  .appended(0)     → List(1000, 100, 1000, 10, 10, 1, 5, 0)
  .sliding(2)      → List(1000,100), List(100,1000), ..., List(5,0)
  .map(adjust)     → 1000, -100, 1000, 10, 10, -1, 5
  .reduce(_ + _)   → 1924
```

In [ ]:
val adjust: List[Int] => Int =
    case List(e1, e2) if e1 < e2 => -e1      // si el siguiente es mayor, resto
    case List(e1, _) => e1

def decode(r: String): Int =
    r.toList
     .map(Sym)
     .appended(0)
     .sliding(2).toList
     .map(adjust)
     .reduce(_ + _)

### `intersect` con `filter` y `exists` (24-25 Móstoles)

In [ ]:
def intersect[A](l1: List[A], l2: List[A]): List[A] =
    l1.filter(e => l2.exists(_ == e))   // me quedo con los de l1 que existen en l2

### `collect` con `flatMap` (25-26 Extraordinaria)

In [ ]:
def collect[A, B](f: A => Option[B], l: List[A]): List[B] =
    l.flatMap(head =>
        f(head) match
            case Some(b) => List(b)    // lo meto
            case None    => Nil        // no meto nada
    )

// atajo de la librería: l.flatMap(f)

### `pair` con `map` y con `flatMap` (23-24 Extraordinaria)

In [ ]:
def pairMap[A, B](e: A, l: List[B]): List[(A, B)] =
    l.map(e2 => (e, e2))

def pairFlatMap[A, B](e: A, l: List[B]): List[(A, B)] =
    l.flatMap(e2 => List((e, e2)))

### `combinaciones2` combinando funciones (23-24 Extraordinaria)

Combinaciones de 2 en 2 sin repetición: para cada elemento, emparejarlo con todos los que
van **detrás**. En ese examen, el anexo define `tails` **sin** incluir la lista entera:

In [ ]:
// tails(List(1,2,3,4)) == List(List(2,3,4), List(3,4), List(4), List())
def tails[A](l: List[A]): List[List[A]] =
    l match
        case Nil => Nil
        case h :: t => t :: tails(t)

```
l                      = List(a, b, c)
tails(l)               = List(List(b,c), List(c), List())
map2(l, tails(l))(pair) = List(List((a,b),(a,c)), List((b,c)), List())
.flatten               = List((a,b), (a,c), (b,c))
```

In [ ]:
def combinaciones2[A](l: List[A]): List[(A, A)] =
    map2(l, tails(l))(pair).flatten

Ojo: el `tails` del examen 23-24 Móstoles **sí** incluye la lista entera
(`tails(List(1)) == List(List(1), Nil)`). Mira siempre la definición que te den.

Versión recursiva equivalente:

In [ ]:
def combinaciones2[A](l: List[A]): List[(A, A)] =
    l match
        case Nil => Nil
        case head :: tail => pair(head, tail) ++ combinaciones2(tail)

### `split` con `filter` y `map` (20-21 Extraordinaria)

In [ ]:
def splitF[A, B](l: List[Either[A, B]]): (List[A], List[B]) =
    (l.filter(_.isLeft).map { case Left(a) => a },
     l.filter(_.isRight).map { case Right(b) => b })

### `sumDivisors` con `filter` (22-23 Extraordinaria)

In [ ]:
def sumDivisors(n: Int): Int =
    (1 to n/2).toList.filter(n % _ == 0).sum

### `lift1st` con `map` (22-23 Móstoles)

`lift1st(f)(l)(b)` aplica `f(_)(b)` a cada elemento de `l`.

In [ ]:
def lift1st[A, B, C](f: A => B => C)(l: List[A])(b: B): List[C] =
    l.map(a => f(a)(b))

def lift2nd[A, B, C](f: A => B => C)(a: A)(l: List[B]): List[C] =
    l.map(f(a))          // f(a) ya es una función B => C

### `op`: `map` + `sequence` + `Option.map` (20-21 Ordinaria)

Dividir todos entre `d` y multiplicar los resultados, solo si todas las divisiones son exactas:

In [ ]:
def op(l: List[Int], d: Int): Option[Int] =
    sequence(                                    // List[Option[Int]] => Option[List[Int]] (archivo 04)
        l.map(n => if d != 0 && n % d == 0 then Some(n / d) else None)
    ).map(_.foldLeft(1)(_ * _))                  // si es Some(lista), multiplico

---

## 6. Consultas sobre datos (exámenes 19-20)

Los exámenes más antiguos traen un "modelo de películas" (`MovieDatabase`, `Film`, `User`,
`Rating`) y funciones básicas (`films(mdb)`, `filmRatings(id)(mdb)`, `getUser(id)(mdb)`…
que devuelven listas). Se piden consultas encadenando `flatMap`, `filter`, `map`,
`sortWith`, `take`, `headOption`. Patrón:

In [ ]:
// Títulos de las películas con todas sus puntuaciones >= 3 (19-20 Prueba)
def topMovies(mdb: MovieDatabase): List[String] =
    films(mdb)
        .filter(film => filmRatings(film.id)(mdb).forall(_.score >= 3))
        .map(_.title)

// Traducir un bucle imperativo (19-20 Diciembre)
// for (rating <- filmRatings(id)(mdb)) for (user <- getUser(rating.user)(mdb)) if (...) out = ... :: out
def misteryHOF(id: Film.Id, year: Int)(mdb: MovieDatabase): List[(String, Int)] =
    filmRatings(id)(mdb)
        .flatMap(rating => getUser(rating.user)(mdb))   // bucle anidado → flatMap
        .filter(_.registered >= year)                   // if → filter
        .map(user => (user.name, user.registered))      // lo que se añade → map

Regla: **cada `for` anidado es un `flatMap`, cada `if` es un `filter`, y lo que se mete en
el resultado es el `map` final.**

---

## 7. Errores típicos

- Usar `map` cuando la función devuelve una lista → te queda `List[List[...]]`. Usa
  `flatMap` (o `map` + `flatten`).
- Olvidar `.toList` después de `sliding`/`grouped`.
- `reduce` en lista vacía da error; si la lista puede estar vacía usa `foldLeft` o `sum`.
- `sortWith(_ < _)` ordena de menor a mayor; `sortWith(_ > _)` de mayor a menor.
- Si el enunciado dice "**utilizando exclusivamente** X, Y, Z", no uses otras (ni recursión).

---

## 8. Ejercicios para practicar

### De exámenes reales

| Ejercicio | Funciones | Examen |
|---|---|---|
| `collect` | `flatMap` | 25-26 Extraordinaria |
| `fromBits` | `reverse`, `zipWithIndex`, `map`, `sum` | 25-26 Extraordinaria |
| `decode` romanos | `map`, `appended`, `sliding`, `reduce` | 24-25 Extraordinaria |
| `intersect` | `filter`, `exists` | 24-25 Móstoles |
| `pair`, `combinaciones2` | `map`, `flatMap`, `map2`, `flatten` | 23-24 Extraordinaria |
| `sumDivisors` | `foldLeft`, `filter` | 22-23 Extraordinaria |
| `lift1st`, `lift2nd` | `map` | 22-23 Móstoles |
| `removeAll` | `filter` | 21-22 Extraordinaria |
| `split`, `components` | `filter`, `map` | 20-21 Extraordinaria |
| `op` | `map`, `sequence` | 20-21 Ordinaria |
| `insertAt` | `take`, `drop` | 19-20 Diciembre |
| Consultas de películas | `flatMap`, `filter`, `map`, `sortWith`, `take` | 19-20 (los tres) |

Más práctica: `PF-3/topic6/6.2 HigherOrderFunctions-exercises.ipynb`.

### Ejercicios nuevos

1. Sumar los cuadrados de los números impares de una lista.
2. `palabrasLargas(l: List[String], n: Int): List[Int]` — longitudes de las palabras de más de n letras.
3. `todosPositivos(l: List[Int]): Boolean` y `algunCero(l: List[Int]): Boolean`.
4. `productoCartesiano[A, B](as: List[A], bs: List[B]): List[(A, B)]` usando `flatMap` y `map`.
5. `posicionesPares[A](l: List[A]): List[A]` — elementos en posiciones 0, 2, 4… usando `zipWithIndex`, `filter` y `map`.

<details>
<summary>Soluciones</summary>

```scala
def ej1(l: List[Int]): Int = l.filter(_ % 2 != 0).map(x => x * x).sum

def palabrasLargas(l: List[String], n: Int): List[Int] = l.filter(_.length > n).map(_.length)

def todosPositivos(l: List[Int]): Boolean = l.forall(_ > 0)
def algunCero(l: List[Int]): Boolean = l.exists(_ == 0)

def productoCartesiano[A, B](as: List[A], bs: List[B]): List[(A, B)] =
    as.flatMap(a => bs.map(b => (a, b)))

def posicionesPares[A](l: List[A]): List[A] =
    l.zipWithIndex.filter((_, i) => i % 2 == 0).map(_._1)
```

</details>

Siguiente: **07-arboles-binarios.ipynb**.